# Build a Tiny Transformer

Implement a complete transformer model from scratch. Build a nanoGPT-style architecture, understand the full forward pass from tokenization through training to text generation. This brings together all previous concepts into a working language model.

[Open this lesson on the site](https://llm.thelittleone.rocks/#/module/tiny-transformer)


## Runtime setup

Before running, set **Runtime -> Change runtime type -> T4 GPU** (or any available GPU). Colab provides PyTorch with CUDA support, so these notebooks run on a real GPU rather than Apple's MPS backend. If a code sample uses `device = torch.device('mps' ...)`, it will fall back to `cpu` on Colab unless you adapt it; replace `'mps'` with `'cuda'` for GPU execution.


In [ ]:
!pip install torch numpy

---

## Lesson examples (optional)

These are the runnable code samples from the lesson sections. Run them to experiment with the concepts before tackling the exercise below.


### Lesson example: Full Transformer Architecture


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class TransformerBlock(nn.Module):
    """Single transformer block: attention + MLP with pre-norm and residuals"""
    def __init__(self, embed_dim, num_heads=8, mlp_ratio=4, dropout=0.1):
        super().__init__()

        # Attention block
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True
        )

        # MLP block
        self.norm2 = nn.LayerNorm(embed_dim)
        mlp_hidden = int(embed_dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, embed_dim),
            nn.Dropout(dropout),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, embed_dim)
            mask: (seq_len, seq_len) causal mask or None

        Returns:
            output: (batch, seq_len, embed_dim)
        """
        # Pre-norm residual: LayerNorm → Attention → Add
        attn_output, _ = self.attention(
            self.norm1(x), self.norm1(x), self.norm1(x),
            attn_mask=mask
        )
        x = x + self.dropout(attn_output)

        # Pre-norm residual: LayerNorm → MLP → Add
        mlp_output = self.mlp(self.norm2(x))
        x = x + self.dropout(mlp_output)

        return x


class NanoGPT(nn.Module):
    """Minimal but complete transformer language model (GPT-style)"""
    def __init__(
        self,
        vocab_size,
        embed_dim=384,
        num_layers=6,
        num_heads=8,
        context_length=256,
        dropout=0.1,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.context_length = context_length

        # Embeddings
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(context_length, embed_dim)
        self.embedding_dropout = nn.Dropout(dropout)

        # Transformer blocks
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio=4, dropout=dropout)
            for _ in range(num_layers)
        ])

        # Output
        self.final_norm = nn.LayerNorm(embed_dim)
        self.output_head = nn.Linear(embed_dim, vocab_size)

        # Register causal mask as buffer
        self.register_buffer(
            "causal_mask",
            torch.tril(torch.ones(context_length, context_length))
        )

    def forward(self, token_ids, targets=None):
        """
        Args:
            token_ids: (batch, seq_len) with token indices
            targets: (batch, seq_len) with target token indices for loss

        Returns:
            logits: (batch, seq_len, vocab_size)
            loss: scalar if targets provided, None otherwise
        """
        batch_size, seq_len = token_ids.shape

        # Embeddings
        token_embeddings = self.token_embedding(token_ids)  # (batch, seq_len, embed_dim)
        positions = torch.arange(seq_len, device=token_ids.device, dtype=torch.long)
        position_embeddings = self.position_embedding(positions)  # (seq_len, embed_dim)

        # Combine embeddings
        x = token_embeddings + position_embeddings  # Broadcasting: (batch, seq_len, embed_dim)
        x = self.embedding_dropout(x)

        # Get causal mask for this sequence length
        mask = self.causal_mask[:seq_len, :seq_len]
        mask = mask.unsqueeze(0)  # (1, seq_len, seq_len)

        # Pass through transformer blocks
        for block in self.transformer_blocks:
            x = block(x, mask=mask)

        # Output projection
        x = self.final_norm(x)
        logits = self.output_head(x)  # (batch, seq_len, vocab_size)

        # Compute loss if targets provided
        loss = None
        if targets is not None:
            # Reshape for cross-entropy: (batch * seq_len, vocab_size) and (batch * seq_len)
            logits_flat = logits.view(-1, self.vocab_size)
            targets_flat = targets.view(-1)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    def generate(self, seed_tokens, max_new_tokens=100, temperature=1.0, top_k=None):
        """
        Generate new tokens given seed.

        Args:
            seed_tokens: (batch, seq_len) initial token IDs
            max_new_tokens: How many tokens to generate
            temperature: Softmax temperature (lower = more deterministic)
            top_k: If provided, sample from top k logits only

        Returns:
            generated: (batch, seq_len + max_new_tokens)
        """
        self.eval()
        generated = seed_tokens.clone()

        with torch.no_grad():
            for _ in range(max_new_tokens):
                # Only feed last context_length tokens
                context = generated[:, -self.context_length:]

                # Get logits for next token
                logits, _ = self(context)

                # Take last position logits
                next_logits = logits[:, -1, :] / temperature

                # Top-k sampling
                if top_k is not None:
                    top_logits, top_indices = torch.topk(next_logits, top_k, dim=-1)
                    next_logits_filtered = torch.full_like(next_logits, float('-inf'))
                    next_logits_filtered.scatter_(1, top_indices, top_logits)
                    next_logits = next_logits_filtered

                # Sample
                probs = F.softmax(next_logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

                generated = torch.cat([generated, next_token], dim=1)

        return generated


# ===== Example Usage =====

vocab_size = 256  # Character-level model
context_length = 64
embed_dim = 128
num_layers = 4

model = NanoGPT(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_layers=num_layers,
    context_length=context_length,
)

# Create dummy data
batch_size = 4
seq_len = 32
token_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
targets = torch.randint(0, vocab_size, (batch_size, seq_len))

# Forward pass with loss
logits, loss = model(token_ids, targets)
print(f"Logits shape: {logits.shape}")  # (4, 32, 256)
print(f"Loss: {loss.item():.4f}")

# Generation
seed = torch.zeros((1, 1), dtype=torch.long)
generated = model.generate(seed, max_new_tokens=50, temperature=0.8)
print(f"\nGenerated shape: {generated.shape}")  # (1, 51)

# Verify gradients
loss.backward()
print(f"✓ Model trained, gradients flowing")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
        

---

## Exercise: Train a Tiny Transformer on Text


Implement a complete NanoGPT model and train it on character-level or token-level text. You must: (1) build the full transformer architecture, (2) create a simple tokenizer, (3) implement training loop with gradient clipping, (4) track train/val loss, (5) generate text samples during training. Train on a small dataset until generating coherent output. This brings together all previous concepts.


In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import math

class CharacterDataset(Dataset):
    """Character-level dataset for language modeling"""
    def __init__(self, text, context_length=64):
        # TODO: Build character vocabulary (unique chars in text)
        # TODO: Create char_to_idx and idx_to_char mappings
        # TODO: Encode entire text to token IDs
        # TODO: Create (input, target) pairs where target is input shifted by 1
        pass

    def __len__(self):
        # TODO: Return number of training examples
        pass

    def __getitem__(self, idx):
        # TODO: Return (token_ids, targets) for this index
        pass

    def decode(self, token_ids):
        # TODO: Convert token IDs back to text using idx_to_char
        pass


class NanoGPT(nn.Module):
    """Transformer language model (GPT-style, minimal but complete)"""
    def __init__(self, vocab_size, embed_dim=128, num_layers=3, context_length=64):
        super().__init__()

        # TODO: Create token embedding
        # TODO: Create positional embedding
        # TODO: Create stack of transformer blocks
        # TODO: Create output projection from embed_dim to vocab_size
        # TODO: Register causal mask as buffer
        pass

    def forward(self, token_ids, targets=None):
        # TODO: Embed tokens and positions
        # TODO: Add positional embeddings to token embeddings
        # TODO: Pass through transformer blocks with causal mask
        # TODO: Project to vocabulary size
        # TODO: Compute cross-entropy loss if targets provided
        pass

    def generate(self, seed_tokens, max_new_tokens=100, temperature=1.0):
        # TODO: Generate new tokens one at a time
        # TODO: Use KV cache or just feed growing sequence
        # TODO: Apply temperature to logits before softmax
        # TODO: Sample next token from softmax distribution
        # TODO: Append to sequence and repeat
        pass


def train():
    # TODO: Load or create text data
    # TODO: Create CharacterDataset
    # TODO: Create DataLoader
    # TODO: Create NanoGPT model
    # TODO: Create optimizer (Adam)
    # TODO: Training loop:
    #   - For each epoch:
    #     - For each batch:
    #       - Forward pass with targets
    #       - Backward pass
    #       - Gradient clipping
    #       - Optimizer step
    #   - Print loss
    #   - Validate on separate set
    #   - Generate sample text
    # TODO: Return trained model
    pass


if __name__ == "__main__":
    model = train()
    print("✓ Training complete!")
      

---

When you're done, head back to the [lesson page](https://llm.thelittleone.rocks/#/module/tiny-transformer) for the solution and discussion.
